# 🛍️ Second Life Commerce — AI Backend
### Run cells in order from top to bottom. Never skip a cell.

---
## CELL 1 — Imports

In [1]:
import httpx
import json
import base64
import os
print("✅ All imports working!")

✅ All imports working!


---
## CELL 2 — API Key Setup
> ⚠️ Paste your NEW Gemini API key below. Never share this key with anyone.

In [ ]:
GEMINI_API_KEY = "you key"
MODEL_URL = "https://generativelanguage.googleapis.com/v1/models/gemini-2.5-flash:generateContent"
print("✅ Key and URL set!")
print(f"Key starts with: {GEMINI_API_KEY[:8]}...")

✅ Key and URL set!
Key starts with: AQ.Ab8RN...


---
## CELL 3 — Core Gemini Functions
> Defines how to talk to Gemini. No API call is made here.

In [33]:
def ask_gemini(prompt_text):
    url = f"{MODEL_URL}?key={GEMINI_API_KEY}"
    payload = {"contents": [{"parts": [{"text": prompt_text}]}]}
    with httpx.Client(verify=False, timeout=30.0) as client:
        response = client.post(url, json=payload)
    result = response.json()
    if "candidates" not in result:
        print("❌ Error from Gemini:", json.dumps(result, indent=2))
        return None
    return result["candidates"][0]["content"]["parts"][0]["text"]

def ask_gemini_with_image(prompt_text, image_path):
    url = f"{MODEL_URL}?key={GEMINI_API_KEY}"
    with open(image_path, "rb") as f:
        image_b64 = base64.b64encode(f.read()).decode()
    ext = image_path.split(".")[-1].lower()
    mime_type = {"jpg": "image/jpeg", "jpeg": "image/jpeg", "png": "image/png"}.get(ext, "image/jpeg")
    payload = {
        "contents": [{
            "parts": [
                {"text": prompt_text},
                {"inline_data": {"mime_type": mime_type, "data": image_b64}}
            ]
        }]
    }
    with httpx.Client(verify=False, timeout=60.0) as client:
        response = client.post(url, json=payload)
    result = response.json()
    if "candidates" not in result:
        print("❌ Error from Gemini:", json.dumps(result, indent=2))
        return None
    return result["candidates"][0]["content"]["parts"][0]["text"]

def safe_parse_json(text):
    text = text.strip()
    if "```json" in text:
        text = text.split("```json")[1].split("```")[0]
    elif "```" in text:
        text = text.split("```")[1].split("```")[0]
    return json.loads(text.strip())

print("✅ Core Gemini functions ready!")

✅ Core Gemini functions ready!


---
## CELL 4 — All Project Functions
> Defines all AI features. No API call is made here.

In [35]:
def check_image_quality(image_path):
    prompt = """Evaluate this image for product condition assessment.
    Check: 1) Lighting, 2) Blur, 3) Angle, 4) Background.
    Return JSON only, no extra text:
    {"quality_ok": true, "issues": [], "retry_message": ""}"""
    result = ask_gemini_with_image(prompt, image_path)
    if result is None:
        return None
    return safe_parse_json(result)

def grade_product_image(image_path):
    prompt = """Analyse this product image for resale condition grading.
    Identify: visible damage, scratches, dents, packaging condition, missing parts, overall wear.
    Return JSON only, no extra text:
    {"grade": "Good", "damage_list": [], "confidence": 85, "damage_locations": []}
    Grade must be one of: Like New, Good, Fair, Poor"""
    result = ask_gemini_with_image(prompt, image_path)
    if result is None:
        return None
    return safe_parse_json(result)

def full_image_analysis(image_path):
    print(f"📸 Analysing: {image_path}")
    quality = check_image_quality(image_path)
    if quality is None:
        return None
    if not quality.get("quality_ok"):
        print(f"❌ Poor image: {quality.get('retry_message')}")
        return {"quality_ok": False, **quality}
    grade_result = grade_product_image(image_path)
    if grade_result is None:
        return None
    return {"quality_ok": True, **grade_result}

def check_confidence(confidence_score, grade_result):
    if confidence_score < 70:
        grade_result["flag_for_human_review"] = True
        grade_result["flag_reason"] = f"Low confidence score: {confidence_score}%. Needs human verification."
    else:
        grade_result["flag_for_human_review"] = False
        grade_result["flag_reason"] = None
    return grade_result

def generate_questionnaire(grade, damage_list):
    prompt = f"""Based on product grade: {grade} and damage list: {damage_list},
    generate 3-5 targeted follow-up questions specific to the damage found.
    Return JSON only, no extra text:
    {{"questions": [{{"id": "q1", "question": "Is the screen functional?", "type": "radio", "options": ["Yes", "No", "Partially"]}}]}}"""
    result = ask_gemini(prompt)
    if result is None:
        return None
    return safe_parse_json(result)

def route_product(category, grade, damage_list, seller_answers):
    prompt = f"""Product: {category}. Grade: {grade}. 
    Damage: {damage_list}. Seller answers: {seller_answers}.
    Determine best destination. Return JSON only, no extra text:
    {{"decision": "resell", "reason": "explanation", "confidence": 90, "flag_for_human_review": false}}
    Decision must be one of: resell, refurbish, donate, recycle"""
    result = ask_gemini(prompt)
    if result is None:
        return None
    result = safe_parse_json(result)
    points_map = {"resell": 100, "donate": 80, "refurbish": 60, "recycle": 40}
    result["green_points_earned"] = points_map.get(result.get("decision"), 0)
    return result

def generate_listing(category, grade, damage_list):
    prompt = f"""Write a resale listing for: Category: {category}, Grade: {grade}, Damage: {damage_list}.
    Return JSON only, no extra text:
    {{"title": "title here", "description": "description here", "suggested_price_inr": 1500, "selling_points": ["point1"]}}"""
    result = ask_gemini(prompt)
    if result is None:
        return None
    return safe_parse_json(result)

def predict_regret(product_category, return_reason):
    prompt = f"""Buyer wants to return {product_category}, reason: '{return_reason}'.
    Return JSON only, no extra text:
    {{"regret_probability": 72, "insight": "one sentence insight", "keep_suggestion": "one tip"}}"""
    result = ask_gemini(prompt)
    if result is None:
        return None
    return safe_parse_json(result)

def calculate_co2(category, decision):
    co2_map = {
        "resell":    {"smartphone": 70, "laptop": 300, "jacket": 20, "shoes": 15, "headphones": 30, "watch": 25, "bag": 18, "book": 5},
        "donate":    {"smartphone": 50, "laptop": 200, "jacket": 15, "shoes": 10, "headphones": 20, "watch": 18, "bag": 12, "book": 3},
        "refurbish": {"smartphone": 60, "laptop": 250, "jacket": 18, "shoes": 12, "headphones": 25, "watch": 20, "bag": 15, "book": 4},
        "recycle":   {"smartphone": 30, "laptop": 100, "jacket": 8,  "shoes": 5,  "headphones": 10, "watch": 10, "bag": 8,  "book": 2}
    }
    kg_saved = co2_map.get(decision, {}).get(category.lower(), 25)
    car_km = round(kg_saved / 0.21, 1)
    return {
        "co2_saved_kg": kg_saved,
        "car_km_equivalent": car_km,
        "message": f"Saving this product avoids {kg_saved}kg CO2 — equal to {car_km}km by car"
    }

def complete_product_pipeline(image_path, category, seller_answers={}):
    print(f"\n{'='*50}")
    print(f"📦 Processing: {category}")
    print(f"{'='*50}")
    analysis = full_image_analysis(image_path)
    if analysis is None or not analysis.get("quality_ok"):
        return {"error": "Image quality too poor", "details": analysis}
    analysis = check_confidence(analysis.get("confidence", 0), analysis)
    questions = generate_questionnaire(analysis.get("grade"), analysis.get("damage_list", []))
    routing = route_product(
        category=category,
        grade=analysis.get("grade"),
        damage_list=analysis.get("damage_list", []),
        seller_answers=seller_answers
    )
    listing = None
    if routing.get("decision") in ["resell", "refurbish"]:
        listing = generate_listing(category=category, grade=analysis.get("grade"), damage_list=analysis.get("damage_list", []))
    co2 = calculate_co2(category, routing.get("decision"))
    return {
        "grade"              : analysis.get("grade"),
        "confidence"         : analysis.get("confidence"),
        "damage_list"        : analysis.get("damage_list"),
        "flag_for_review"    : analysis.get("flag_for_human_review"),
        "flag_reason"        : analysis.get("flag_reason"),
        "routing_decision"   : routing.get("decision"),
        "routing_reason"     : routing.get("reason"),
        "green_points_earned": routing.get("green_points_earned"),
        "co2_saved_kg"       : co2.get("co2_saved_kg"),
        "car_km_equivalent"  : co2.get("car_km_equivalent"),
        "co2_message"        : co2.get("message"),
        "questions"          : questions.get("questions", []) if questions else [],
        "listing"            : listing
    }

print("✅ All project functions ready!")

✅ All project functions ready!


---
## CELL 5 — Quick Gemini Test
> First actual API call. Confirm Gemini responds before anything else.

In [37]:
reply = ask_gemini("Say hello in one sentence")
print(reply)
print("✅ Gemini working!")

Hello!
✅ Gemini working!


---
## CELL 6 — Test Image Analysis
> Put a product image in your secondlife-backend folder and update the filename below.

In [11]:
# ✏️ Change this to your actual image filename
result = full_image_analysis("your_new_image.jpg")

if result:
    print("\n📊 IMAGE ANALYSIS RESULT:")
    print(f"Quality OK      : {result.get('quality_ok')}")
    print(f"Grade           : {result.get('grade')}")
    print(f"Confidence      : {result.get('confidence')}%")
    print(f"Damages Found   : {result.get('damage_list')}")
    print(f"Damage Locations: {result.get('damage_locations')}")
    print("✅ Image analysis working!")

📸 Analysing: your_new_image.jpg

📊 IMAGE ANALYSIS RESULT:
Quality OK      : True
Grade           : Like New
Confidence      : 95%
Damages Found   : []
Damage Locations: []
✅ Image analysis working!


---
## CELL 7 — Test Questionnaire Generator

In [39]:
questions = generate_questionnaire("Fair", ["scratch on back", "minor dent on corner"])

print("📋 Generated Questions:")
for q in questions["questions"]:
    print(f"\n→ {q['question']}")
    if q.get("options"):
        print(f"   Options: {q['options']}")
print("✅ Questionnaire working!")

📋 Generated Questions:

→ Does the scratch on the back penetrate beyond the surface, or is it purely cosmetic?
   Options: ['Purely Cosmetic', 'Penetrates Surface']

→ Does the minor dent on the corner cause any displacement or cracking of the screen or frame?
   Options: ['Yes', 'No']

→ Does the minor dent on the corner impede the functionality of any buttons or ports in that area?
   Options: ['Yes', 'No']

→ Does either the scratch on the back or the dent on the corner expose any internal components?
   Options: ['Yes', 'No']
✅ Questionnaire working!


---
## CELL 8 — Test Routing Engine

In [41]:
routing = route_product(
    category="Smartphone",
    grade="Good",
    damage_list=["small scratch on back cover"],
    seller_answers={"functional": "Yes", "age_months": "18", "reason": "Upgrading"}
)

print("🔀 Routing Decision:")
print(f"→ Decision     : {routing['decision'].upper()}")
print(f"→ Reason       : {routing['reason']}")
print(f"→ Confidence   : {routing['confidence']}%")
print(f"→ Green Points : {routing['green_points_earned']} pts")
print(f"→ Human Review : {routing['flag_for_human_review']}")
print("✅ Routing working!")

🔀 Routing Decision:
→ Decision     : RESELL
→ Reason       : The smartphone is fully functional ('Yes'), relatively new (18 months old), and has only a minor cosmetic defect (a small scratch on the back cover) that does not impede its usability or significantly diminish its value. It can be sold as 'used - good' condition.
→ Confidence   : 95%
→ Green Points : 100 pts
→ Human Review : False
✅ Routing working!


---
## CELL 9 — Test Listing Generator

In [43]:
listing = generate_listing("Smartphone", "Good", ["small scratch on back"])

print("🏷️ Generated Listing:")
print(f"Title          : {listing['title']}")
print(f"Description    : {listing['description']}")
print(f"Suggested Price: ₹{listing['suggested_price_inr']}")
print(f"Selling Points : {listing['selling_points']}")
print("✅ Listing generator working!")

🏷️ Generated Listing:
Title          : Smartphone - Good Condition - Minor Back Scratch - Great Value!
Description    : Up for sale is a functional smartphone in good overall condition. This device is perfect for everyday basic use, a reliable secondary phone, or a budget-friendly option for a first-time user. The screen is clear and free from significant scratches or cracks. Please note there is a small, cosmetic scratch on the back casing of the phone. All core functions work perfectly, including calling, messaging, and app usage. Battery health is good for daily tasks. Grab a dependable smartphone at an unbeatable price!
Suggested Price: ₹1500
Selling Points : ['Good overall condition', 'Fully functional for daily use', 'Clear display', 'Reliable and budget-friendly', 'Ready for immediate use']
✅ Listing generator working!


---
## CELL 10 — Test Regret Predictor

In [45]:
regret = predict_regret("Bluetooth Headphones", "Sound quality not as expected")

print("🧠 Regret Predictor:")
print(f"→ Regret Probability : {regret['regret_probability']}%")
print(f"→ Insight            : {regret['insight']}")
print(f"→ Suggestion         : {regret['keep_suggestion']}")
print("✅ Regret predictor working!")

🧠 Regret Predictor:
→ Regret Probability : 72%
→ Insight            : The buyer's subjective listening experience did not align with their personal expectations for audio performance.
→ Suggestion         : Suggest adjusting the equalizer settings on their connected device to better suit their audio preferences.
✅ Regret predictor working!


---
## CELL 11 — Test CO2 Calculator

In [47]:
co2 = calculate_co2("smartphone", "resell")
print(f"CO2 Saved     : {co2['co2_saved_kg']} kg")
print(f"Car Equivalent: {co2['car_km_equivalent']} km")
print(f"Message       : {co2['message']}")
print("✅ CO2 calculator working!")

CO2 Saved     : 70 kg
Car Equivalent: 333.3 km
Message       : Saving this product avoids 70kg CO2 — equal to 333.3km by car
✅ CO2 calculator working!


---
## CELL 12 — Test Confidence Flag

In [49]:
test_result = {"grade": "Fair", "confidence": 65, "damage_list": ["scratch"]}
flagged = check_confidence(65, test_result)
print(f"Flag for Review : {flagged['flag_for_human_review']}")
print(f"Flag Reason     : {flagged['flag_reason']}")
print("✅ Confidence threshold working!")

Flag for Review : True
Flag Reason     : Low confidence score: 65%. Needs human verification.
✅ Confidence threshold working!


---
## CELL 13 — Test Full Pipeline with One Image
> This runs the complete flow: image → grade → route → listing → CO2

In [51]:
# ✏️ Change filename and category to match your image
result = complete_product_pipeline("your_new_image.jpg", "Smartphone")

print("\n📊 COMPLETE PIPELINE RESULT:")
print(f"Grade           : {result.get('grade')}")
print(f"Confidence      : {result.get('confidence')}%")
print(f"Damages         : {result.get('damage_list')}")
print(f"Flag for Review : {result.get('flag_for_review')}")
print(f"Routing Decision: {result.get('routing_decision')}")
print(f"Routing Reason  : {result.get('routing_reason')}")
print(f"Green Points    : {result.get('green_points_earned')} pts")
print(f"CO2 Saved       : {result.get('co2_saved_kg')} kg")
print(f"Car Equivalent  : {result.get('car_km_equivalent')} km")
print(f"CO2 Message     : {result.get('co2_message')}")
print(f"Questions Count : {len(result.get('questions', []))}")
if result.get('listing'):
    print(f"Listing Title   : {result['listing'].get('title')}")
    print(f"Suggested Price : ₹{result['listing'].get('suggested_price_inr')}")
print("✅ Full pipeline working!")


📦 Processing: Smartphone
📸 Analysing: your_new_image.jpg

📊 COMPLETE PIPELINE RESULT:
Grade           : Like New
Confidence      : 95%
Damages         : []
Flag for Review : False
Routing Decision: resell
Routing Reason  : The smartphone is in 'Like New' condition with no reported damage, indicating it is fully functional and aesthetically excellent, making it suitable for immediate resale at a premium price.
Green Points    : 100 pts
CO2 Saved       : 70 kg
Car Equivalent  : 333.3 km
CO2 Message     : Saving this product avoids 70kg CO2 — equal to 333.3km by car
Questions Count : 4
Listing Title   : Premium Smartphone - Like New, Flawless Condition!
Suggested Price : ₹35000
✅ Full pipeline working!


---
## CELL 14 — Test 10 Product Images
> Add 10 product images to your secondlife-backend folder. Update filenames and categories below.

In [ ]:
import time

test_products = [
    ("your_image_1.jpg", "Smartphone"),
    ("your_image_2.jpg", "Laptop"),
    ("your_image_3.jpg", "Shoes"),
    ("your_image_4.jpg", "Jacket"),
    ("your_image_5.jpg", "Headphones"),
    ("your_image_6.jpg", "Watch"),
    ("your_image_7.jpg", "Bag"),
    ("your_image_8.jpg", "Book"),
    ("your_image_9.jpg", "Smartphone"),
    ("your_image_10.jpg", "Laptop"),
]

results = []

for i, (image_path, category) in enumerate(test_products):
    print(f"\n🔄 Testing: {category} — {image_path}")
    result = complete_product_pipeline(image_path, category)
    results.append({"product": category, "image": image_path, **result})
    print(f"✅ Grade: {result.get('grade')} | Decision: {result.get('routing_decision')} | CO2: {result.get('co2_saved_kg')}kg | Points: {result.get('green_points_earned')}")
    
    # Wait 60 seconds between each product to avoid quota limit
    if i < len(test_products) - 1:
        print(f"⏳ Waiting 60 seconds before next image...")
        time.sleep(60)

print(f"\n{'='*50}")
print(f"✅ All {len(results)} products tested!")
print(f"{'='*50}")

---
## CELL 15 — Start FastAPI Server
> ⚠️ Run this LAST. Once running, open http://localhost:8000/docs in your browser.
> Share http://localhost:8000 with your teammates for frontend connection.

In [ ]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

nest_asyncio.apply()

app = FastAPI(title="Second Life Commerce API")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class QuestionnaireRequest(BaseModel):
    grade: str
    damage_list: list

class RoutingRequest(BaseModel):
    category: str
    grade: str
    damage_list: list
    seller_answers: dict

class ListingRequest(BaseModel):
    category: str
    grade: str
    damage_list: list

class RegretRequest(BaseModel):
    product_category: str
    return_reason: str

class CO2Request(BaseModel):
    category: str
    decision: str

@app.get("/")
def home():
    return {"message": "✅ Second Life Commerce API is running!"}

@app.post("/api/grade")
async def grade_endpoint(file: UploadFile = File(...)):
    image_bytes = await file.read()
    temp_path = f"temp_{file.filename}"
    with open(temp_path, "wb") as f:
        f.write(image_bytes)
    result = full_image_analysis(temp_path)
    os.remove(temp_path)
    return result

@app.post("/api/questionnaire")
def questionnaire_endpoint(data: QuestionnaireRequest):
    return generate_questionnaire(data.grade, data.damage_list)

@app.post("/api/route-product")
def routing_endpoint(data: RoutingRequest):
    return route_product(data.category, data.grade, data.damage_list, data.seller_answers)

@app.post("/api/generate-listing")
def listing_endpoint(data: ListingRequest):
    return generate_listing(data.category, data.grade, data.damage_list)

@app.post("/api/regret-predict")
def regret_endpoint(data: RegretRequest):
    return predict_regret(data.product_category, data.return_reason)

@app.post("/api/co2-impact")
def co2_endpoint(data: CO2Request):
    return calculate_co2(data.category, data.decision)

print("🚀 Server starting at http://localhost:8000")
print("📖 Test all endpoints at http://localhost:8000/docs")
print("👥 Share http://localhost:8000 with your teammates")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
import asyncio
asyncio.get_event_loop().run_until_complete(server.serve())